# KCD 2026 · 개선된 클러스터링

KCD_ONE.ipynb를 이미 실행한 PC에서, **그 노트북과 같은 폴더**에 두고 위에서부터 실행합니다. 원자료를 다시 읽지 않고 1단계 산출물(`kcd_work/run_all/full/kcd_liver_v2_outputs`)만 씁니다. 10분 안팎 걸립니다.

바뀐 점: 군집 수 2-12, 대각·완전 공분산 비교, R 비 제외(ALT·ALP와 중복), AST 추가, 선택 규칙 고정(부트스트랩 ARI ≥ 0.75이고 BIC가 최저에서 1% 안인 가장 작은 K), 정리된 모든 변수로 군집 해석, 군집별 예후, 군집이 예후 정보를 더하는지 검정.

끝나면 노트북 옆에 `kcd_cluster_package_*.zip`(집계와 그림만, 10 미만 칸 가림)이 생깁니다.

In [ ]:
# ===== 설정 =====
from pathlib import Path
OUT = Path.cwd() / 'kcd_work' / 'run_all' / 'full' / 'kcd_liver_v2_outputs'   # 1단계 산출물 폴더
N_BOOT = 30
print('산출물 폴더 확인:', OUT.exists(), OUT)

In [ ]:
# ===== 코드 풀기 (수정하지 마세요) =====
import base64, sys
CODE = Path.cwd() / 'kcd_cluster_code' / 'kcd_liver'
CODE.mkdir(parents=True, exist_ok=True)
(CODE / '__init__.py').write_text('', encoding='utf-8')
for name, b in {'pattern.py': 'IiIiTGl2ZXIgaW5qdXJ5IHBhdHRlcm4gKGZpeGVkIGNhdGVnb3JpZXMpIGFuZCBhbiBleHBsb3JhdG9yeSBjbHVzdGVyaW5nIGNoZWNrLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKClBBVFRFUk5TID0gWyJoZXBhdG9jZWxsdWxhciIsICJtaXhlZCIsICJjaG9sZXN0YXRpYyIsICJpc29sYXRlZF9iaWxpcnViaW4iLCAidW5jbGFzc2lmaWFibGUiXQpTVEFCTEVfQVJJID0gMC43NQoKCmRlZiByX3BhdHRlcm4oYWx0X3JhdGlvLCBhbHBfcmF0aW8sIHRiaWxfcmF0aW89Tm9uZSwgcl9yYXRpbz1Ob25lLCBhYm5vcm1hbD0yLjApOgogICAgIiIiQUNHIDIwMTc6IFIgPSAoQUxUL1VMTikvKEFMUC9VTE4pLiBSPj01IGhlcGF0b2NlbGx1bGFyLCBSPD0yIGNob2xlc3RhdGljLCBlbHNlIG1peGVkLiIiIgogICAgZiA9IGxhbWJkYSB2OiBOb25lIGlmIHYgaXMgTm9uZSBvciAoaXNpbnN0YW5jZSh2LCBmbG9hdCkgYW5kIG1hdGguaXNuYW4odikpIGVsc2UgZmxvYXQodikKICAgIGFsdCwgYWxwLCB0YiwgciA9IGYoYWx0X3JhdGlvKSwgZihhbHBfcmF0aW8pLCBmKHRiaWxfcmF0aW8pLCBmKHJfcmF0aW8pCiAgICBpZiByIGlzIE5vbmUgYW5kIGFsdCBpcyBub3QgTm9uZSBhbmQgYWxwOgogICAgICAgIHIgPSBhbHQgLyBhbHAKICAgIGVuenltZV9hYm5vcm1hbCA9IChhbHQgaXMgbm90IE5vbmUgYW5kIGFsdCA+PSBhYm5vcm1hbCkgb3IgKGFscCBpcyBub3QgTm9uZSBhbmQgYWxwID49IGFibm9ybWFsKQogICAgaWYgZW56eW1lX2Fibm9ybWFsIGFuZCByIGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiAiaGVwYXRvY2VsbHVsYXIiIGlmIHIgPj0gNSBlbHNlICJjaG9sZXN0YXRpYyIgaWYgciA8PSAyIGVsc2UgIm1peGVkIgogICAgaWYgdGIgaXMgbm90IE5vbmUgYW5kIHRiID49IGFibm9ybWFsOgogICAgICAgIHJldHVybiAiaXNvbGF0ZWRfYmlsaXJ1YmluIgogICAgcmV0dXJuICJ1bmNsYXNzaWZpYWJsZSIKCgpkZWYgcGF0dGVybl9mcmFtZShmZWF0dXJlczogcGQuRGF0YUZyYW1lKSAtPiBwZC5TZXJpZXM6CiAgICBnID0gbGFtYmRhIGM6IGZlYXR1cmVzW2NdIGlmIGMgaW4gZmVhdHVyZXMgZWxzZSBwZC5TZXJpZXMobnAubmFuLCBpbmRleD1mZWF0dXJlcy5pbmRleCkKICAgIHJldHVybiBwZC5TZXJpZXMoW3JfcGF0dGVybihhLCBiLCBjLCBkKSBmb3IgYSwgYiwgYywgZCBpbiB6aXAoZygiQUxUX3JhdGlvIiksIGcoIkFMUF9yYXRpbyIpLCBnKCJUQklMX3JhdGlvIiksIGcoIlJfcmF0aW8iKSldLAogICAgICAgICAgICAgICAgICAgICBpbmRleD1mZWF0dXJlcy5pbmRleCwgbmFtZT0ibGl2ZXJfcGF0dGVybiIpCgoKZGVmIF9wcmVwX3BhcmFtcyhkZjogcGQuRGF0YUZyYW1lLCBjb2xzKToKICAgICIiIlByZXByb2Nlc3NpbmcgbGVhcm5lZCBvbiBkZXZlbG9wbWVudCBkYXRhIG9ubHkgKGxvZzIgY2hvaWNlLCBtZWRpYW4sIG1lYW4sIFNEKS4iIiIKICAgIFggPSBkZltjb2xzXS5hcHBseShwZC50b19udW1lcmljLCBlcnJvcnM9ImNvZXJjZSIpCiAgICBsb2djID0gW2MgZm9yIGMgaW4gY29scyBpZiBsZW4oWFtjXS5kcm9wbmEoKSkgYW5kIChYW2NdLmRyb3BuYSgpID4gMCkuYWxsKCkgYW5kIFhbY10ubnVuaXF1ZSgpID4gMl0KICAgIGZvciBjIGluIGxvZ2M6CiAgICAgICAgWFtjXSA9IG5wLmxvZzIoWFtjXS5jbGlwKGxvd2VyPTAuMDUpKQogICAgbWVkID0gWC5tZWRpYW4oKQogICAgWCA9IFguZmlsbG5hKG1lZCkKICAgIHJldHVybiB7ImNvbHMiOiBsaXN0KGNvbHMpLCAibG9nX2NvbHMiOiBsb2djLCAibWVkaWFuIjogbWVkLnRvX2RpY3QoKSwgIm1lYW4iOiBYLm1lYW4oKS50b19kaWN0KCksCiAgICAgICAgICAgICJzZCI6IFguc3RkKGRkb2Y9MCkucmVwbGFjZSgwLCAxKS50b19kaWN0KCl9CgoKZGVmIF9hcHBseV9wYXJhbXMoZGY6IHBkLkRhdGFGcmFtZSwgUDogZGljdCkgLT4gbnAubmRhcnJheToKICAgIFggPSBkZi5yZWluZGV4KGNvbHVtbnM9UFsiY29scyJdKS5hcHBseShwZC50b19udW1lcmljLCBlcnJvcnM9ImNvZXJjZSIpCiAgICBmb3IgYyBpbiBQWyJsb2dfY29scyJdOgogICAgICAgIFhbY10gPSBucC5sb2cyKFhbY10uY2xpcChsb3dlcj0wLjA1KSkKICAgIFggPSBYLmZpbGxuYShwZC5TZXJpZXMoUFsibWVkaWFuIl0pKQogICAgcmV0dXJuICgoWCAtIHBkLlNlcmllcyhQWyJtZWFuIl0pKSAvIHBkLlNlcmllcyhQWyJzZCJdKSkudG9fbnVtcHkoKQoKCmRlZiBfbWF0cml4KGRmOiBwZC5EYXRhRnJhbWUsIGNvbHMpOgogICAgcmV0dXJuIF9hcHBseV9wYXJhbXMoZGYsIF9wcmVwX3BhcmFtcyhkZiwgY29scykpCgoKZGVmIGFzc2lnbl9jbHVzdGVycyhtb2RlbDogZGljdCwgZGY6IHBkLkRhdGFGcmFtZSkgLT4gbnAubmRhcnJheToKICAgICIiIkFwcGx5IGEgc2F2ZWQgY2x1c3RlcmluZyBtb2RlbCAoZnJvbSBnbW1fZXhwbG9yZShyZXR1cm5fbW9kZWw9VHJ1ZSkpIHRvIG5ldyBkYXRhLCBlLmcuIGFub3RoZXIgaG9zcGl0YWwuIiIiCiAgICByZXR1cm4gbW9kZWxbImdtbSJdLnByZWRpY3QoX2FwcGx5X3BhcmFtcyhkZiwgbW9kZWxbInByZXAiXSkpCgoKZGVmIGdtbV9leHBsb3JlKGRmOiBwZC5EYXRhRnJhbWUsIGNvbHMsIGtzPXJhbmdlKDIsIDkpLCBuX2Jvb3Q9NTAsIHNlZWQ9MCwgbWluX249MzAsIG1pbl9vYnNlcnZlZD0wLjUsIHJldHVybl9tb2RlbD1GYWxzZSk6CiAgICAiIiJTdXBwbGVtZW50YXJ5OiBCSUMtc2VsZWN0ZWQgR2F1c3NpYW4gbWl4dHVyZSBwbHVzIGJvb3RzdHJhcCBBUkkgc3RhYmlsaXR5LiIiIgogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGFkanVzdGVkX3JhbmRfc2NvcmUKICAgIGZyb20gc2tsZWFybi5taXh0dXJlIGltcG9ydCBHYXVzc2lhbk1peHR1cmUKICAgIGNvbHMgPSBbYyBmb3IgYyBpbiBjb2xzIGlmIGMgaW4gZGYgYW5kIHBkLnRvX251bWVyaWMoZGZbY10sIGVycm9ycz0iY29lcmNlIikubm90bmEoKS5tZWFuKCkgPj0gbWluX29ic2VydmVkXQogICAgb3V0ID0geyJtZXRob2QiOiAiR2F1c3NpYW5NaXh0dXJlKGRpYWcpLCBsb2cyIGZvciBwb3NpdGl2ZSBsYWJzLCBLIGJ5IEJJQywgYm9vdHN0cmFwIEFSSSIsCiAgICAgICAgICAgIm4iOiBpbnQobGVuKGRmKSksICJmZWF0dXJlcyI6IGNvbHMsICJzdGFibGVfdGhyZXNob2xkIjogU1RBQkxFX0FSSX0KICAgIGlmIGxlbihkZikgPCBtaW5fbiBvciBub3QgY29sczoKICAgICAgICByZXR1cm4gKHsqKm91dCwgInN0YXR1cyI6ICJpbnN1ZmZpY2llbnQiLCAidXNlZCI6IEZhbHNlfSwgTm9uZSkgKyAoKE5vbmUsKSBpZiByZXR1cm5fbW9kZWwgZWxzZSAoKSkKICAgIHByZXAgPSBfcHJlcF9wYXJhbXMoZGYsIGNvbHMpCiAgICBYID0gX2FwcGx5X3BhcmFtcyhkZiwgcHJlcCkKICAgIGZpdCA9IGxhbWJkYSBrLCBBLCBzOiBHYXVzc2lhbk1peHR1cmUoaywgY292YXJpYW5jZV90eXBlPSJkaWFnIiwgbl9pbml0PTMsIHJhbmRvbV9zdGF0ZT1zKS5maXQoQSkKICAgIGJpYyA9IFtdCiAgICBmb3IgayBpbiBrczoKICAgICAgICBpZiBrID49IGxlbihYKToKICAgICAgICAgICAgYnJlYWsKICAgICAgICBiaWMuYXBwZW5kKHsibl9jbHVzdGVycyI6IGludChrKSwgImJpYyI6IGZsb2F0KGZpdChrLCBYLCBzZWVkKS5iaWMoWCkpfSkKICAgIGtfYmVzdCA9IG1pbihiaWMsIGtleT1sYW1iZGEgcjogclsiYmljIl0pWyJuX2NsdXN0ZXJzIl0KICAgIHJlZl9tb2RlbCA9IGZpdChrX2Jlc3QsIFgsIHNlZWQpCiAgICByZWYgPSByZWZfbW9kZWwucHJlZGljdChYKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBhcmlzID0gW10KICAgIGZvciBiIGluIHJhbmdlKG5fYm9vdCk6CiAgICAgICAgaWR4ID0gcm5nLmludGVnZXJzKDAsIGxlbihYKSwgbGVuKFgpKQogICAgICAgIGFyaXMuYXBwZW5kKGFkanVzdGVkX3JhbmRfc2NvcmUocmVmLCBmaXQoa19iZXN0LCBYW2lkeF0sIHNlZWQgKyBiICsgMSkucHJlZGljdChYKSkpCiAgICBxID0gbnAucXVhbnRpbGUoYXJpcywgWy4yNSwgLjUsIC43NV0pCiAgICBzdGFibGUgPSBib29sKHFbMV0gPj0gU1RBQkxFX0FSSSkKICAgIG91dC51cGRhdGUoc3RhdHVzPSJvayIsIGJpYz1iaWMsIGtfc2VsZWN0ZWQ9aW50KGtfYmVzdCksIGFyaV9xMjU9ZmxvYXQocVswXSksIGFyaV9tZWRpYW49ZmxvYXQocVsxXSksCiAgICAgICAgICAgICAgIGFyaV9xNzU9ZmxvYXQocVsyXSksIG5fYm9vdD1uX2Jvb3QsIHVzZWQ9c3RhYmxlLAogICAgICAgICAgICAgICBzaXplcz1baW50KHMpIGZvciBzIGluIG5wLmJpbmNvdW50KHJlZiwgbWlubGVuZ3RoPWtfYmVzdCldKQogICAgaWYgcmV0dXJuX21vZGVsOgogICAgICAgIHJldHVybiBvdXQsIHJlZiwgeyJnbW0iOiByZWZfbW9kZWwsICJwcmVwIjogcHJlcCwgImsiOiBpbnQoa19iZXN0KSwgInN0YWJsZSI6IHN0YWJsZSwgImtpbmQiOiAiZ21tX2NsdXN0ZXIifQogICAgcmV0dXJuIG91dCwgcmVmCg==', 'cluster_report.py': '...'}.items():
    (CODE / name).write_bytes(base64.b64decode(b))
sys.path.insert(0, str(CODE.parent))
for m in [k for k in sys.modules if k.startswith('kcd_liver')]:
    del sys.modules[m]
from kcd_liver import cluster_report as cr
DEMO_IDS = set(int(x) for x in 'subject_id\n10000032\n10001217\n10001725\n10002428\n10002495\n10002930\n10003046\n10003400\n10004235\n10004422\n10004457\n10004720\n10004733\n10005348\n10005817\n10005866\n10005909\n10006053\n10006580\n10007058\n10007795\n10007818\n10007928\n10008287\n10008454\n10009035\n10009049\n10009628\n10010471\n10010867\n10011398\n10012552\n10012853\n10013049\n10014078\n10014354\n10014729\n10015272\n10015860\n10015931\n10016150\n10016742\n10016810\n10017492\n10018081\n10018328\n10018423\n10018501\n10018845\n10019003\n10019172\n10019385\n10019568\n10019777\n10019917\n10020187\n10020306\n10020640\n10020740\n10020786\n10020944\n10021118\n10021312\n10021487\n10021666\n10021938\n10022017\n10022041\n10022281\n10022880\n10023117\n10023239\n10023771\n10024043\n10025463\n10025612\n10026255\n10026354\n10026406\n10027445\n10027602\n10029291\n10029484\n10031404\n10031757\n10032725\n10035185\n10035631\n10036156\n10037861\n10037928\n10037975\n10038081\n10038933\n10038992\n10038999\n10039708\n10039831\n10039997\n10040025\n'.split() if x.strip().isdigit())
print('코드 준비 완료')

In [ ]:
# ===== 입력 읽기 =====
import json, time
import pandas as pd
feat = pd.read_csv(OUT / '08_features.csv', low_memory=False)
outc = pd.read_csv(OUT / '10_outcomes_EVALUATION_ONLY.csv', low_memory=False)
ev = pd.read_csv(OUT / '05_events_private.csv', low_memory=False, usecols=lambda c: c in {'case_id', 'subject_id', 'group'})
extra = ev[['case_id', 'group']].copy() if 'group' in ev else ev[['case_id']].copy()
pred = OUT / '09_agent_predictions.jsonl'
if pred.exists():
    fl, conds = [], {}
    for line in pred.read_text(encoding='utf-8').splitlines():
        r = json.loads(line)
        a = r.get('answer') or {}
        flags = a.get('flags') or next((v.get('flags') for v in a.values() if isinstance(v, dict) and v.get('flags')), None) or {}
        conds[r.get('condition')] = conds.get(r.get('condition'), 0) + 1
        if flags:
            fl.append({'case_id': r.get('case_id'), 'base': r.get('condition', 'base') == 'base',
                       **{f'flag_{k}': float(bool((v or {}).get('present'))) for k, v in flags.items()}})
    print('규칙 판정 기록:', conds, '· 원인 플래그가 있는 기록', len(fl))
    if fl:
        fl = pd.DataFrame(fl).sort_values('base', ascending=False).drop(columns='base').to_dict('records')
        extra = extra.merge(pd.DataFrame(fl).drop_duplicates('case_id'), on='case_id', how='left')
exclude = set(ev.loc[ev.subject_id.isin(DEMO_IDS), 'case_id']) if 'subject_id' in ev else set()
print(f'첫 사건 {len(feat):,}건, 결과 {len(outc):,}건, 공개 데모 제외 {len(exclude)}건, 추가 변수 {extra.shape[1] - 1}개')

In [ ]:
# ===== 군집 분석 =====
t = time.time()
R = cr.run(feat, outc, extra=extra, exclude_cases=exclude, n_boot=N_BOOT, log=print)
print(f'완료 {(time.time() - t) / 60:.1f}분 · 선택:', {k: R['selected'][k] for k in ['covariance', 'k', 'ari_median']})
for p in R['profiles']:
    print(f"C{p['cluster']} {p['name']} n={p['n_dev']} · 주요 변수:", [cr.var_label(v) for v in R['characterization']['top_by_cluster'].get(p['cluster'], [])])

In [ ]:
# ===== 그림과 묶음 =====
import zipfile
from datetime import datetime
from IPython.display import Image, display
FIG = OUT / 'cluster_v2'
FIG.mkdir(exist_ok=True)
cr.save(R, FIG / 'cluster_report.json')
files = cr.render(R, FIG)
for p in files:
    display(Image(filename=str(p)))
zp = Path.cwd() / f"kcd_cluster_package_{datetime.now():%Y%m%d_%H%M}.zip"
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(FIG / 'cluster_report.json', 'cluster_report.json')
    for p in files:
        z.write(p, 'figures/' + p.name)
print('돌려줄 파일:', zp)